Project part 1:
Shirel Amar 207065103 & Sarah Derhy 340889435
https://github.com/SarahDerhy/Data-analyse-advanced-in-Python

In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import pandas as pd
from pathlib import Path
import re
import io
from tqdm import tqdm

In [2]:
#Importing our tabs directly from the IMDB website

In [3]:
BASE_URL = "https://datasets.imdbws.com/"

folders = {
    "basics":     "title.basics.tsv.gz",
    "ratings":    "title.ratings.tsv.gz",
    "principals": "title.principals.tsv.gz",
    "names":      "name.basics.tsv.gz"
}

def download_file(folder_name):
    url = BASE_URL + folder_name
    response = requests.get(url, stream=True)
    return io.BytesIO(response.content)

basics_file     = download_file(folders["basics"])
ratings_file    = download_file(folders["ratings"])
principals_file = download_file(folders["principals"])
names_file      = download_file(folders["names"])

In [4]:
#The data is too big, so we cut it into chunks and import it chunk by chunk

In [5]:
basics_chunks = []
for chunk in pd.read_csv(basics_file,
    sep="\t",
    na_values="\\N",
    on_bad_lines="skip",
    compression="gzip",
    low_memory=False,
    chunksize=100_000):
    
    chunk = chunk[(chunk["titleType"] == "movie") & 
                  (chunk["primaryTitle"].str.startswith(("K", "k", "N", "n"), na=False))]
    basics_chunks.append(chunk)

basics = pd.concat(basics_chunks, ignore_index=True)

In [6]:
basics.head()

,tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
0,tt0000984,movie,Niños en la alameda,Niños en la alameda,0,1909.0,NaN,NaN,NaN
1,tt0001277,movie,Kapergasten,Kapergasten,0,1910.0,NaN,NaN,Drama
2,tt0001911,movie,Nell Gwynne,Sweet Nell of Old Drury,0,1911.0,NaN,50.0,"Biography,Drama,History"
3,tt0003042,movie,Kleiner Svend und seine Mutter,Kleiner Svend und seine Mutter,0,1913.0,NaN,NaN,Drama
4,tt0003218,movie,Novillada de beneficencia en Zaragoza,Novillada de beneficencia en Zaragoza,0,1913.0,NaN,NaN,NaN


In [7]:
basics_tconst = set(basics["tconst"]) 
rating_chunks = []
for chunk in pd.read_csv(ratings_file,
    sep="\t",
    na_values="\\N",
    on_bad_lines="skip",
    compression="gzip",
    low_memory=False,
    chunksize=100_000):
    
    chunk = chunk[chunk["tconst"].isin(basics_tconst)]
    rating_chunks.append(chunk)

ratings = pd.concat(rating_chunks, ignore_index=True)


In [8]:
ratings.head()

,tconst,averageRating,numVotes
0,tt0000984,3.7,17
1,tt0001277,4.1,31
2,tt0001911,3.7,29
3,tt0004391,5.4,73
4,tt0005589,5.9,39


In [9]:
principals_chunks = []
for chunk in pd.read_csv(principals_file,
    sep="\t",
    na_values="\\N",
    on_bad_lines="skip",
    compression="gzip",
    low_memory=False,
    chunksize=100_000):
    
    chunk = chunk[chunk["tconst"].isin(basics_tconst)]
    principals_chunks.append(chunk)

principals = pd.concat(principals_chunks, ignore_index=True)


In [10]:
principals.head()

,tconst,ordering,nconst,category,job,characters
0,tt0000984,1,nm0023107,director,NaN,NaN
1,tt0001277,1,nm0169878,actor,NaN,"[""Salomon Baadsmand""]"
2,tt0001277,2,nm0299757,actor,NaN,"[""Landsforræder""]"
3,tt0001277,3,nm0736379,actress,NaN,"[""Ung Jomfru""]"
4,tt0001277,4,nm0772788,actor,NaN,"[""Kapergasten""]"


In [11]:
principals_nconst = set(principals["nconst"])

names_chunks = []
for chunk in pd.read_csv(names_file,
    sep="\t",
    na_values="\\N",
    on_bad_lines="skip",
    compression="gzip",
    low_memory=False,
    chunksize=100_000):
    
    chunk = chunk[chunk["nconst"].isin(principals_nconst)]
    names_chunks.append(chunk)

names = pd.concat(names_chunks, ignore_index=True)

In [12]:
names.head()

,nconst,primaryName,birthYear,deathYear,primaryProfession,knownForTitles
0,nm0000001,Fred Astaire,1899.0,1987.0,"actor,miscellaneous,producer","tt0072308,tt0050419,tt0027125,tt0025164"
1,nm0000002,Lauren Bacall,1924.0,2014.0,"actress,miscellaneous,soundtrack","tt0037382,tt0075213,tt0038355,tt0045891"
2,nm0000003,Brigitte Bardot,1934.0,2025.0,"actress,music_department,producer","tt0057345,tt0049189,tt0056404,tt0054452"
3,nm0000004,John Belushi,1949.0,1982.0,"actor,writer,music_department","tt0072562,tt0077975,tt0080455,tt0078723"
4,nm0000006,Ingrid Bergman,1915.0,1982.0,"actress,producer,soundtrack","tt0034583,tt0038109,tt0036855,tt0038787"


In [13]:
#Tabs are loaded, we drop the column we need.

In [14]:
basics= basics.drop(columns=["originalTitle", "isAdult","endYear"])
principals= principals.drop(columns=["ordering","job","characters"])
names= names[["nconst"]]

In [15]:
#We keep in "principals" only "actor" and "actress" .

In [16]:
principals= principals[principals["category"].isin(["actor", "actress"])]

In [17]:
#We merge our tabs together.

In [18]:
principals_names = principals.merge(names, on="nconst", how="left")

In [19]:
df=basics.merge(ratings, on="tconst", how="left")

In [20]:
df= df.merge(principals_names, on="tconst", how="left")

In [21]:
#Adding more filter on our data.

In [22]:
df= df[df["startYear"] <= 2024]
df["runtimeMinutes"]= pd.to_numeric(df["runtimeMinutes"], errors="coerce")
df= df[(df["runtimeMinutes"] >= 60) & (df["runtimeMinutes"] <= 300)]

In [23]:
df.sample()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,nconst,category
426,tt0009420,movie,Nattliga toner,1918.0,61.0,"Drama,Romance",6.2,52.0,nm0803601,actress


In [24]:
#Creating a new column: "lead_actors_ids".

In [25]:
top5 = (df[df["category"].isin(["actor", "actress"])]
               .groupby("tconst")["nconst"]
               .apply(lambda x: list(x)[:5])
               .reset_index()
               .rename(columns={"nconst": "lead_actors_ids"}))
df = df.merge(top5, on="tconst", how="left")

In [26]:
df.sample()

,tconst,titleType,primaryTitle,startYear,runtimeMinutes,genres,averageRating,numVotes,nconst,category,lead_actors_ids
2368,tt0026567,movie,Kid Courageous,1935.0,62.0,"Drama,Western",6.5,48.0,nm0184704,actor,"[nm0824496, nm0096154, nm0517425, nm0184704, n..."


In [27]:
df['tconst'].nunique()

21266

We reduced our dataset from 21,265 to approximately 6000 movies by keeping only movies with at least 100 votes, as they are more likely to have information available on Wikipedia.

In [28]:
df = df[df["numVotes"] >= 100]

In [44]:
df['tconst'].nunique()

6458

In [30]:
#For now we have multiple rows for the same movie. 
#Since the "lead_actor_ids" column already groups the actors, we can remove duplicates and keep one row per movie.

In [31]:
df_unique = df.drop_duplicates(subset="tconst")

In [32]:
#We start from now the websrapping from Wikipedia to complete our data set.

In [33]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0 Safari/537.36"
}

In [34]:
#We have a clean function to have our final results written in a clean way.

In [35]:
def clean(value):
    if value:
        value = value.replace("\xa0", " ")
        value = re.sub(r'\[.*?\]', '', value)
        return value.strip()
    return None



In [36]:
#normalize function: normalize title by removing symbols and splitting into set of lowercase words for comparison
#get_movie_info function: Search on Wikipedia by title and year, extract language, country, budget and box office from the infobox and the plot from the page, return all as a dictionary or None if not found.

In [37]:
def normalize(text):
    return set(re.sub(r"[^\w\s]", "", text.lower()).split())

def get_movie_info(primaryTitle, year):
    empty = {
        "language": None,
        "country": None,
        "plot": None,
        "budget": None,
        "boxOffice": None
    }

    try:
        search_url = "https://en.wikipedia.org/w/api.php"
        search_params = {
            "action": "query",
            "list": "search",
            "srsearch": f"{primaryTitle} {year} film".strip() if year else f"{primaryTitle} film",
            "srnamespace": "0",
            "srlang": "en",
            "format": "json"
        }

        response = requests.get(search_url, params=search_params, headers=headers, timeout=10)

        if response.status_code == 429:
            wait = int(response.headers.get("Retry-After", 60))
            print(f"Rate limited. Wait {wait}s...")
            time.sleep(wait)
            return get_movie_info(primaryTitle, year)

        if response.status_code != 200:
            return empty

        data = response.json()

        if not data["query"]["search"]:
            return empty

        title_words = normalize(primaryTitle)
        best_with_year = None
        best_without_year = None

        for result in data["query"]["search"]:
            page_words = normalize(result["title"])
            page_title_lower = result["title"].lower()

            title_match = title_words.issubset(page_words)
            year_match = year and str(year) in page_title_lower

            if title_match and year_match and best_with_year is None:
                best_with_year = result["title"]

            if title_match and not year_match and best_without_year is None:
                best_without_year = result["title"]

        if best_with_year:
            page_title = best_with_year
        elif best_without_year:
            page_title = best_without_year
        else:
            return empty

        wiki_url = "https://en.wikipedia.org/wiki/" + page_title.replace(" ", "_")

        page_response = requests.get(wiki_url, headers=headers, timeout=10)

        if page_response.status_code == 429:
            wait = int(page_response.headers.get("Retry-After", 60))
            print(f"Rate limited. Wait {wait}s...")
            time.sleep(wait)
            return get_movie_info(primaryTitle, year)

        if page_response.status_code != 200:
            return empty

        soup = BeautifulSoup(page_response.content, "html.parser")

        infobox = soup.find("table", {"class": "infobox"})
        language = None
        country = None
        budget = None
        box_office = None

        if infobox:
            for row in infobox.find_all("tr"):
                header = row.find("th")
                value = row.find("td")

                if header and value:
                    header_text = header.get_text().strip().lower()
                    value_text = value.get_text().strip()

                    if "language" in header_text:
                        language = value_text
                    elif "countr" in header_text:     #Countr to get country AND countries
                        country = value_text
                    elif "budget" in header_text:
                        budget = value_text
                    elif "box office" in header_text or "boxoffice" in header_text:
                        box_office = value_text

        plot = None
        plot_header = soup.find(
            lambda tag: tag.name in ["h2", "h3"] and "plot" in tag.get_text().lower()
        )

        if plot_header:
            next_p = plot_header.find_next("p")

            if next_p:
                plot = next_p.get_text().strip()

        time.sleep(1.5)

        return {
            "language": clean(language),
            "country": clean(country),
            "plot": plot,
            "budget": clean(budget),
            "boxOffice": clean(box_office)
        }

    except Exception as e:
        print(f"Error for '{primaryTitle}': {e}")
        return empty

In [40]:
#We run the get_movies_info function by chunk to not reach the rate limite from Wikipedia.

In [43]:
chunk_size = 50
chunks = [df_unique[i:i+chunk_size] for i in range(0, len(df_unique), chunk_size)]

all_results = []
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i+1}/{len(chunks)} ---")
    
    for _, row in tqdm(chunk.iterrows(), total=len(chunk)):
        info = get_movie_info(row["primaryTitle"], row["startYear"])
        info["tconst"] = row["tconst"]
        all_results.append(info)
    
    pd.DataFrame(all_results).to_csv("wikipedia_results_checkpoint.csv", index=False)
    print(f"Chunked saved nº{i+1} ({len(all_results)} films processed)")
    
    if i < len(chunks) - 1:
        print(f"10sec break before the next chunk.")
        time.sleep(10)

wikipedia_df = pd.DataFrame(all_results)
final_df = df_unique.merge(wikipedia_df, on="tconst", how="left")
final_df.to_csv("final_df.csv", index=False)
print("Scraping complete.")


--- Chunk 1/130 ---


 30%|████████████████████████▌                                                         | 15/50 [01:00<02:22,  4.08s/it]

Rate limited. Wait 17s...


 50%|█████████████████████████████████████████                                         | 25/50 [02:13<02:19,  5.58s/it]

Rate limited. Wait 4s...


...

Rate limited. Wait 46s...


 64%|████████████████████████████████████████████████████▍                             | 32/50 [15:47<07:32, 25.15s/it]

Rate limited. Wait 48s...


 64%|████████████████████████████████████████████████████▍                             | 32/50 [16:37<09:21, 31.18s/it]


KeyboardInterrupt: 

In [ ]:
final_df.sample()

In [ ]:
#Checking how many null values we have in each column.

In [ ]:
final_df.isna().sum()